In [ ]:
/

In [ ]:
import hashlib

def sha256(data):
    """Calculates the SHA-256 hash of the input data."""
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

class MerkleTree:
    """A simple implementation of a Merkle Tree."""

    def __init__(self, data_blocks):
        """Initializes the Merkle Tree with a list of data blocks."""
        self.data_blocks = data_blocks
        self.tree = self._build_tree(data_blocks)

    def _build_tree(self, blocks):
        """Recursively builds the Merkle tree."""
        if len(blocks) == 1:
            return [sha256(blocks[0])]

        if len(blocks) % 2 != 0:
            blocks.append(blocks[-1]) # Duplicate last block if odd number

        new_level = []
        for i in range(0, len(blocks), 2):
            combined_hash = sha256(blocks[i] + blocks[i+1])
            new_level.append(combined_hash)

        return self._build_tree(new_level) + new_level

    def get_root(self):
        """Returns the Merkle root hash."""
        return self.tree[0]

    def get_proof(self, data_block):
        """Generates a Merkle proof for a given data block."""
        if data_block not in self.data_blocks:
            return None

        index = self.data_blocks.index(data_block)
        proof = []
        level_size = len(self.data_blocks)
        level_start_index = len(self.tree) - level_size

        while level_size > 1:
            is_right_node = index % 2 != 0
            sibling_index = index - 1 if is_right_node else index + 1

            # Handle the case where the last block was duplicated
            if sibling_index >= level_start_index + level_size:
                 sibling_index = index

            sibling_hash = self.tree[level_start_index + sibling_index]
            proof.append((sibling_hash, is_right_node))

            index //= 2
            level_size //= 2
            level_start_index -= level_size if level_size > 0 else 0

        return proof

    def verify_proof(self, data_block, proof, root):
        """Verifies a Merkle proof for a given data block against the root hash."""
        current_hash = sha256(data_block)
        for sibling_hash, is_right_node in proof:
            if is_right_node:
                current_hash = sha256(sibling_hash + current_hash)
            else:
                current_hash = sha256(current_hash + sibling_hash)
        return current_hash == root

# Example Usage
data = ["block A", "block B", "block C", "block D", "block E"]
merkle_tree = MerkleTree(data)

root_hash = merkle_tree.get_root()
print(f"Merkle Root: {root_hash}")

# Get proof for a data block
block_to_verify = "block C"
proof = merkle_tree.get_proof(block_to_verify)

if proof:
    print(f"\nProof for '{block_to_verify}': {proof}")
    # Verify the proof
    is_valid = merkle_tree.verify_proof(block_to_verify, proof, root_hash)
    print(f"Is proof valid? {is_valid}")

    # Verify a tampered block (should be invalid)
    tampered_block = "block X"
    is_valid_tampered = merkle_tree.verify_proof(tampered_block, proof, root_hash)
    print(f"Is proof valid for tampered block '{tampered_block}'? {is_valid_tampered}")

else:
    print(f"\n'{block_to_verify}' not found in data blocks.")

Merkle Root: 2d2d901934643e47e6fb844dacb0a89ebee5afe4a648df17a2aa9a41ddb61509

Proof for 'block C': [('9dfb6458d82337e3c6fe41841a7eb84c98147071e3637a295470b84991394536', False), ('5b551312db9336add74fc5c4099415f75f7901a5745604b4cb3f378731162a5d', True)]
Is proof valid? False
Is proof valid for tampered block 'block X'? False


# Task
Generate code to simulate a simple blockchain structure using Merkle trees.

## Define a block class

### Subtask:
Create a class to represent a block in the blockchain. Each block should contain data (like transactions), a timestamp, the hash of the previous block, and its own hash (calculated using the Merkle root of its data).


**Reasoning**:
The subtask is to create a `Block` class with attributes for timestamp, data, previous hash, Merkle root, and its own hash, along with a method to calculate the block's hash. This can be done in a single code block.



In [ ]:
import time

class Block:
    """Represents a block in the blockchain."""

    def __init__(self, timestamp, data, previous_hash):
        """Initializes a Block instance."""
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        self.merkle_root = None  # Will be set later with Merkle tree
        self.hash = None # Will be set later after calculating

    def calculate_hash(self):
        """Calculates and returns the hash of the block."""
        # This will be implemented later using timestamp, previous_hash, and merkle_root
        pass

## Define a blockchain class

### Subtask:
Create a class to manage the chain of blocks. This class should handle adding new blocks, calculating their hashes, and ensuring the chain's integrity.


**Reasoning**:
Define the `Blockchain` class with an empty chain and methods for adding blocks and checking validity.



In [ ]:
class Blockchain:
    """Manages the chain of blocks."""

    def __init__(self):
        """Initializes the Blockchain with an empty chain."""
        self.chain = []

    def add_block(self, new_block):
        """Appends a new block to the chain."""
        self.chain.append(new_block)

    def is_valid(self):
        """Checks the integrity of the blockchain."""
        # Placeholder for integrity check logic
        pass

## Integrate merkle trees

### Subtask:
Modify the Block class to use the `MerkleTree` class to calculate the Merkle root of the block's data. This Merkle root will be included in the block's hash calculation.


**Reasoning**:
Modify the `Block` class to incorporate the `MerkleTree` for calculating the Merkle root and update the `calculate_hash` method to use the Merkle root in the hash calculation.



In [ ]:
import time

class Block:
    """Represents a block in the blockchain."""

    def __init__(self, timestamp, data, previous_hash):
        """Initializes a Block instance."""
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        # Calculate Merkle root using the MerkleTree class
        merkle_tree = MerkleTree(self.data)
        self.merkle_root = merkle_tree.get_root()
        self.hash = self.calculate_hash() # Calculate the block's hash upon initialization

    def calculate_hash(self):
        """Calculates and returns the hash of the block."""
        # Concatenate timestamp, previous hash, and Merkle root
        hash_string = str(self.timestamp) + str(self.previous_hash) + str(self.merkle_root)
        # Calculate SHA-256 hash
        return sha256(hash_string)


## Implement proof-of-work (optional but recommended for a more realistic simulation)

### Subtask:
Add a simple proof-of-work mechanism to the Blockchain class to control the rate at which new blocks can be added and make tampering more difficult.


**Reasoning**:
Modify the Blockchain and Block classes to include the proof-of-work mechanism as described in the instructions.



In [ ]:
import time

class Block:
    """Represents a block in the blockchain."""

    def __init__(self, timestamp, data, previous_hash):
        """Initializes a Block instance."""
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        # Calculate Merkle root using the MerkleTree class
        merkle_tree = MerkleTree(self.data)
        self.merkle_root = merkle_tree.get_root()
        self.nonce = 0 # Add nonce attribute
        self.hash = self.calculate_hash() # Calculate the block's hash upon initialization

    def calculate_hash(self):
        """Calculates and returns the hash of the block."""
        # Concatenate timestamp, previous hash, Merkle root, and nonce
        hash_string = str(self.timestamp) + str(self.previous_hash) + str(self.merkle_root) + str(self.nonce)
        # Calculate SHA-256 hash
        return sha256(hash_string)

class Blockchain:
    """Manages the chain of blocks."""

    def __init__(self, difficulty=4): # Add difficulty attribute
        """Initializes the Blockchain with an empty chain and difficulty."""
        self.chain = []
        self.difficulty = difficulty

    def add_block(self, new_block):
        """Appends a new block to the chain."""
        self.chain.append(new_block)

    def is_valid(self):
        """Checks the integrity of the blockchain."""
        # Placeholder for integrity check logic
        pass

    def mine_block(self, block):
        """Mines a block by finding a hash that meets the difficulty requirement."""
        target_prefix = '0' * self.difficulty
        while not block.hash.startswith(target_prefix):
            block.nonce += 1
            block.hash = block.calculate_hash()
        print(f"Block mined: {block.hash}")
        self.add_block(block)


## Add block validation

### Subtask:
Implement a method in the Blockchain class to validate the integrity of the entire chain by checking the hashes of each block and the link to the previous block.


**Reasoning**:
Implement the `is_valid` method in the `Blockchain` class to check the integrity of the chain by verifying block hashes and the previous hash links.



In [ ]:
class Block:
    """Represents a block in the blockchain."""

    def __init__(self, timestamp, data, previous_hash):
        """Initializes a Block instance."""
        self.timestamp = timestamp
        self.data = data
        self.previous_hash = previous_hash
        # Calculate Merkle root using the MerkleTree class
        merkle_tree = MerkleTree(self.data)
        self.merkle_root = merkle_tree.get_root()
        self.nonce = 0 # Add nonce attribute
        self.hash = self.calculate_hash() # Calculate the block's hash upon initialization

    def calculate_hash(self):
        """Calculates and returns the hash of the block."""
        # Concatenate timestamp, previous hash, Merkle root, and nonce
        hash_string = str(self.timestamp) + str(self.previous_hash) + str(self.merkle_root) + str(self.nonce)
        # Calculate SHA-256 hash
        return sha256(hash_string)

class Blockchain:
    """Manages the chain of blocks."""

    def __init__(self, difficulty=4): # Add difficulty attribute
        """Initializes the Blockchain with an empty chain and difficulty."""
        self.chain = []
        self.difficulty = difficulty

    def add_block(self, new_block):
        """Appends a new block to the chain."""
        self.chain.append(new_block)

    def is_valid(self):
        """Checks the integrity of the blockchain."""
        # Check if the chain is not empty
        if not self.chain:
            return True # An empty chain can be considered valid

        # Iterate through the blocks, starting from the second block
        for i in range(1, len(self.chain)):
            current_block = self.chain[i]
            previous_block = self.chain[i-1]

            # Check if the current block's previous_hash matches the actual hash of the previous block
            if current_block.previous_hash != previous_block.hash:
                print(f"Block {i} previous hash mismatch.")
                return False

            # Recalculate the current block's hash and compare it to the stored hash
            if current_block.hash != current_block.calculate_hash():
                print(f"Block {i} hash recalculation mismatch.")
                return False

            # (Optional) Verify the proof of work for each block (except the genesis block)
            target_prefix = '0' * self.difficulty
            if not current_block.hash.startswith(target_prefix):
                print(f"Block {i} proof of work invalid.")
                return False


        return True # If all checks pass, the chain is valid

    def mine_block(self, block):
        """Mines a block by finding a hash that meets the difficulty requirement."""
        target_prefix = '0' * self.difficulty
        while not block.hash.startswith(target_prefix):
            block.nonce += 1
            block.hash = block.calculate_hash()
        print(f"Block mined: {block.hash}")
        self.add_block(block)


## Simulate adding blocks

### Subtask:
Simulate adding blocks by creating instances of the Block and Blockchain classes and adding blocks to the chain with some sample data.


**Reasoning**:
Simulate adding blocks by creating instances of the Block and Blockchain classes and adding blocks to the chain with some sample data.



In [ ]:
# 1. Create an instance of the Blockchain class.
blockchain = Blockchain(difficulty=2) # Lower difficulty for faster mining in simulation

# 2. Create the genesis block
genesis_block = Block(time.time(), ["Genesis Block Data"], "0")

# 3. Mine the genesis block
blockchain.mine_block(genesis_block)

# 4. Create several more Block instances and 5. Mine each of these subsequent blocks
block2 = Block(time.time(), ["Transaction 1", "Transaction 2"], blockchain.chain[-1].hash)
blockchain.mine_block(block2)

block3 = Block(time.time(), ["Transaction 3"], blockchain.chain[-1].hash)
blockchain.mine_block(block3)

block4 = Block(time.time(), ["Transaction 4", "Transaction 5", "Transaction 6"], blockchain.chain[-1].hash)
blockchain.mine_block(block4)

# Verify the chain (optional but good practice)
print("\nIs the blockchain valid?", blockchain.is_valid())

Block mined: 00f565c74a530179457ba3ab5d109c83a4b100e97b5f9425a0402185dbd7a35b
Block mined: 00e2df8dead3f97fe5c18113c252d90eb2719577c4c30a1e1722684e07dd28b1
Block mined: 004da2e8bf6254e070cf6db57046c33718d6f0b76a66192b7ee8b6f45f102080
Block mined: 00f9607013d4b75b58e610c8bc4e694f7770835752f61eedb9571fa9fe376c54

Is the blockchain valid? True


## Verify the blockchain

### Subtask:
Verify the blockchain to show that it's intact.


**Reasoning**:
Call the is_valid method on the blockchain instance and print the result.



In [ ]:
# 1. Call the is_valid() method on the blockchain instance created in the previous step.
is_valid = blockchain.is_valid()

# 2. Print the result of the is_valid() method to the console
print("\nIs the blockchain valid?", is_valid)


Is the blockchain valid? True


# Task
Collect data from different LLMs using their APIs.

## Identify llms and apis

### Subtask:
Determine which LLMs you want to collect data from and identify their corresponding APIs.


## Set up api access

### Subtask:
Obtain necessary API keys and set up authentication for each LLM.


## Install necessary libraries

### Subtask:
Install Python libraries required to interact with the APIs (e.g., `openai`, `transformers`, `google-generativeai`).


**Reasoning**:
Install the required libraries for interacting with the LLM APIs using pip.



In [9]:
%pip install openai google-generativeai transformers

## Write data collection code

### Subtask:
Write code to make requests to each LLM's API and retrieve the desired data.


**Reasoning**:
Write functions to interact with each LLM API as described in the instructions, including retrieving API keys from environment variables, making API calls, and handling responses and errors.



In [10]:
import os
import openai
import google.generativeai
import requests

def get_openai_completion(prompt):
    """Makes a request to the OpenAI API and returns the generated text."""
    try:
        openai_api_key = os.getenv("OPENAI_API_KEY")
        if not openai_api_key:
            raise ValueError("OPENAI_API_KEY environment variable not set.")

        client = openai.OpenAI(api_key=openai_api_key)

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error with OpenAI API: {e}")
        return None

def get_gemini_completion(prompt):
    """Makes a request to the Google Gemini API and returns the generated text."""
    try:
        google_api_key = os.getenv("GOOGLE_API_KEY")
        if not google_api_key:
            raise ValueError("GOOGLE_API_KEY environment variable not set.")

        google.generativeai.configure(api_key=google_api_key)

        model = google.generativeai.GenerativeModel('gemini-pro')
        response = model.generate_content(prompt)

        return response.text.strip()
    except Exception as e:
        print(f"Error with Google Gemini API: {e}")
        return None

def get_huggingface_completion(prompt, model_id="gpt2"):
    """Makes a request to the Hugging Face Inference API and returns the generated text."""
    try:
        huggingface_api_key = os.getenv("HF_API_KEY")
        if not huggingface_api_key:
            raise ValueError("HF_API_KEY environment variable not set.")

        api_url = f"https://api-inference.huggingface.co/models/{model_id}"
        headers = {"Authorization": f"Bearer {huggingface_api_key}"}

        response = requests.post(api_url, headers=headers, json={"inputs": prompt})
        response.raise_for_status() # Raise an exception for bad status codes

        result = response.json()
        if isinstance(result, list) and result and 'generated_text' in result[0]:
            return result[0]['generated_text'].strip()
        else:
            print(f"Unexpected response format from Hugging Face API: {result}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error with Hugging Face API request: {e}")
        return None
    except Exception as e:
        print(f"Error with Hugging Face API: {e}")
        return None


## Process and store data

### Subtask:
Process the collected data as needed and store it in a suitable format (e.g., CSV, JSON).


**Reasoning**:
Implement the data collection function, define sample prompts, call the data collection function, create a pandas DataFrame, and save it to a CSV file.



In [11]:
import pandas as pd

def collect_data(prompts):
    """Collects responses from different LLMs for a list of prompts."""
    results = []
    for prompt in prompts:
        openai_response = get_openai_completion(prompt)
        gemini_response = get_gemini_completion(prompt)
        huggingface_response = get_huggingface_completion(prompt)

        results.append({
            "prompt": prompt,
            "openai_response": openai_response,
            "gemini_response": gemini_response,
            "huggingface_response": huggingface_response
        })
    return results

# Define a list of sample prompts
sample_prompts = [
    "What is the capital of France?",
    "Explain the concept of photosynthesis in simple terms.",
    "Write a short poem about autumn.",
    "What are the benefits of exercise?"
]

# Call the collect_data function
collected_data = collect_data(sample_prompts)

# Create a Pandas DataFrame
df = pd.DataFrame(collected_data)

# Save the DataFrame to a CSV file
df.to_csv("llm_responses.csv", index=False)

# Print a confirmation message
print("Data saved to llm_responses.csv")

Error with OpenAI API: OPENAI_API_KEY environment variable not set.
Error with Google Gemini API: GOOGLE_API_KEY environment variable not set.
Error with Hugging Face API: HF_API_KEY environment variable not set.
Error with OpenAI API: OPENAI_API_KEY environment variable not set.
Error with Google Gemini API: GOOGLE_API_KEY environment variable not set.
Error with Hugging Face API: HF_API_KEY environment variable not set.
Error with OpenAI API: OPENAI_API_KEY environment variable not set.
Error with Google Gemini API: GOOGLE_API_KEY environment variable not set.
Error with Hugging Face API: HF_API_KEY environment variable not set.
Error with OpenAI API: OPENAI_API_KEY environment variable not set.
Error with Google Gemini API: GOOGLE_API_KEY environment variable not set.
Error with Hugging Face API: HF_API_KEY environment variable not set.
Data saved to llm_responses.csv


## Handle rate limits and errors

### Subtask:
Implement error handling and consider rate limits for API calls.


**Reasoning**:
Modify the existing API call functions to include more specific error handling for rate limits and other API-related exceptions, and implement a retry mechanism with exponential backoff for rate limit errors. Add a small delay after each API call.



In [12]:
import os
import openai
import google.generativeai
import requests
import time
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5), retry=retry_if_exception_type(openai.APIStatusError))
def get_openai_completion(prompt):
    """Makes a request to the OpenAI API and returns the generated text with error handling and retry."""
    try:
        openai_api_key = os.getenv("OPENAI_API_KEY")
        if not openai_api_key:
            raise ValueError("OPENAI_API_KEY environment variable not set.")

        client = openai.OpenAI(api_key=openai_api_key)

        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ]
        )
        time.sleep(1) # Add a small delay after the API call
        return response.choices[0].message.content.strip()
    except openai.APIStatusError as e:
        print(f"OpenAI API Status Error: {e.status_code} - {e.response}")
        raise # Re-raise the exception for tenacity to handle retries
    except Exception as e:
        print(f"Error with OpenAI API: {e}")
        return None

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5), retry=retry_if_exception_type(google.api_core.exceptions.ResourceExhausted))
def get_gemini_completion(prompt):
    """Makes a request to the Google Gemini API and returns the generated text with error handling and retry."""
    try:
        google_api_key = os.getenv("GOOGLE_API_KEY")
        if not google_api_key:
            raise ValueError("GOOGLE_API_KEY environment variable not set.")

        google.generativeai.configure(api_key=google_api_key)

        model = google.generativeai.GenerativeModel('gemini-pro')
        response = model.generate_content(prompt)

        time.sleep(1) # Add a small delay after the API call
        return response.text.strip()
    except google.api_core.exceptions.ResourceExhausted as e:
        print(f"Google Gemini API Resource Exhausted (Rate Limit): {e}")
        raise # Re-raise the exception for tenacity to handle retries
    except Exception as e:
        print(f"Error with Google Gemini API: {e}")
        return None

@retry(wait=wait_exponential(multiplier=1, min=4, max=10), stop=stop_after_attempt(5), retry=retry_if_exception_type(requests.exceptions.RequestException))
def get_huggingface_completion(prompt, model_id="gpt2"):
    """Makes a request to the Hugging Face Inference API and returns the generated text with error handling and retry."""
    try:
        huggingface_api_key = os.getenv("HF_API_KEY")
        if not huggingface_api_key:
            raise ValueError("HF_API_KEY environment variable not set.")

        api_url = f"https://api-inference.huggingface.co/models/{model_id}"
        headers = {"Authorization": f"Bearer {huggingface_api_key}"}

        response = requests.post(api_url, headers=headers, json={"inputs": prompt})
        response.raise_for_status() # Raise an exception for bad status codes

        result = response.json()
        if isinstance(result, list) and result and 'generated_text' in result[0]:
            time.sleep(1) # Add a small delay after the API call
            return result[0]['generated_text'].strip()
        else:
            print(f"Unexpected response format from Hugging Face API: {result}")
            return None

    except requests.exceptions.RequestException as e:
        print(f"Error with Hugging Face API request: {e}")
        raise # Re-raise the exception for tenacity to handle retries
    except Exception as e:
        print(f"Error with Hugging Face API: {e}")
        return None


## Summary:

### Data Analysis Key Findings

* The task involved collecting data from OpenAI's GPT-3.5, Google's Gemini, and a Hugging Face text generation model using their respective APIs.
* API documentation, base URLs, relevant endpoints, and authentication methods were identified for each LLM.
* The necessary Python libraries (`openai`, `google-generativeai`, `transformers`) were confirmed to be pre-installed in the environment.
* Code was written to make requests to each API and retrieve generated text based on input prompts.
* Error handling and rate limit considerations were implemented for API calls, including using the `tenacity` library for retries with exponential backoff and adding small delays between calls.
* A Pandas DataFrame was intended to be created from the collected data and saved to a CSV file named "llm\_responses.csv".
* Due to missing API keys in the environment variables, the actual API calls within the data collection step failed, although the script for data processing and saving to CSV completed successfully.

### Insights or Next Steps

* The primary next step is to set the necessary environment variables (`OPENAI_API_KEY`, `GOOGLE_API_KEY`, `HF_API_KEY`) with valid API keys to enable successful data collection from the LLMs.
* After setting up the API keys, the data collection and processing step should be re-executed to obtain actual responses from the LLMs and populate the "llm\_responses.csv" file with the collected data.
